# Agentic-SDK 舊版展示 Notebook

這份 Notebook 保留早期展示用範例，方便快速檢查 SDK 匯入、基本 workflow 與 LLM 連線。主線教學請優先參考 01 到 06。

## 1. 環境建置

首先，我們從 GitHub 複製專案原始碼並切換到專案目錄中。

In [ ]:
!pip install -q "git+https://github.com/R300-AI/Agentic-SDK.git"

## 2. 驗證 SDK 安裝

嘗試導入 `agentic_sdk` 以確認安裝是否成功。

In [ ]:
import agentic_sdk
print('Agentic SDK import ok')

## 3. Workflow 基本架構範例

這個範例展示了 SDK 的核心邏輯：
1. **Perceive**: 接收使用者輸入。
2. **Retrieve**: 透過關鍵字檢索預設的知識庫內容。
3. **Action**: 直接輸出檢索到的結果。

我們將測試一個關於「TSiP」定義的問題。

In [ ]:
# 公開介面範例
from agentic_sdk import Workflow
from agentic_sdk.modules import DirectAnswerAction, KeywordRetrieve, PassThroughPerceive

workflow = Workflow(
    perceive=PassThroughPerceive(),
    retrieve=KeywordRetrieve(
        items=[
            {
                "keywords": ["agentic sdk", "sdk"],
                "content": "Agentic SDK 是一個以 workflow 組裝 agent 行為的 Python library。",
            },
            {
                "keywords": ["tsip"],
                "content": "TSiP 是工研院主導的國產 AI 晶片落地藍圖。",
            },
        ],
    ),
    action=DirectAnswerAction(),
)

result = workflow.run("TSiP 是什麼？")
print(result.final_message)

## 4. LLM 模型連線測試

這一段會直接呼叫 OpenAI 相容的 API 端點，確認模型能正常回應。請先在執行環境設定 `AGENTIC_SDK_OPENAI_API_KEY`；如果你只想看 SDK 基本流程，可以先跳過這一段。

In [ ]:
import os
from openai import OpenAI

endpoint = os.environ.get(
    'AGENTIC_SDK_OPENAI_ENDPOINT',
    'https://agentic-sdk-foundry.cognitiveservices.azure.com/openai/v1/',
)
deployment = os.environ.get('AGENTIC_SDK_OPENAI_DEPLOYMENT', 'agentic-sdk-gpt-5.4')
api_key = os.environ.get('AGENTIC_SDK_OPENAI_API_KEY')

if not api_key:
    raise RuntimeError('請先設定 AGENTIC_SDK_OPENAI_API_KEY，再執行模型連線測試。')

client = OpenAI(
    base_url=endpoint,
    api_key=api_key,
)

response = client.chat.completions.create(
    model=deployment,
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user', 'content': 'I am going to Paris, what should I see?'},
    ],
    max_completion_tokens=16384,
)

print(response.choices[0].message.content)

## 5. 進階應用：結合 LLM 的生成式行動 (Generative Action)

最後，我們將 Workflow 中的 `Action` 模組替換為 `GenerativeAction`。這會讓 Agent 不只是複製檢索到的內容，而是能根據 `system_prompt` 的指示，以更自然、更具結構化的方式回答使用者的問題。

In [ ]:
from agentic_sdk import Workflow
from agentic_sdk.modules import GenerativeAction, KeywordRetrieve, PassThroughPerceive

workflow = Workflow(
    perceive=PassThroughPerceive(),
    retrieve=KeywordRetrieve(
        items=[
            {
                "keywords": ["tsip"],
                "content": "TSiP 是工研院主導的國產 AI 晶片落地藍圖。",
            },
        ],
    ),
    action=GenerativeAction(
        base_url=endpoint,
        api_key=api_key,
        model=deployment,
        system_prompt=(
            "你是 Agentic SDK demo 的 Action 模組。請根據 retrieved_context 用自然語氣回答，"
            "不要逐字照抄，也不要加入 retrieved_context 沒有的外部事實。"
            "請輸出兩句繁體中文：第一句回答問題，第二句用『簡單說，』開頭做一句補充說明。"
        )
    ),
)

result = workflow.run("TSiP 是什麼？")
print(result.final_message)